In [ ]:
using JLD2
using Plots
using Statistics
using Printf

gr()


In [ ]:
cutoffs = ["1.0e-4", "1.0e-5", "1.0e-6", "1.0e-7", "1.0e-8", "1.0e-9", "1.0e-10", "1.0e-11", "1.0e-12", "1.0e-13", "1.0e-14"]
sigma_list = [0.2, 0.02, 0.002, 0.0002, 0.0]
sigma_list_small = [0.02, 0.002, 0.0002, 0.0]




data_dict = Dict()
results = Dict()

for c in cutoffs
    filename = joinpath(@__DIR__, "truncation_$(c).jld2")
    
    if isfile(filename)
        data = JLD2.load(filename)
        data_dict[c] = data
        
        results[c] = data["entanglement_results"]
        var_name = "results_" * replace(c, "-" => "_", "." => "_")
        eval(:($(Symbol(var_name)) = $(results[c])))
        
    else
        println("Warning: File $filename not found.")
    end
end

In [ ]:
"""
Calculates the deviation index for every truncation and every sigma.
Identifies the first index 'i' where disordered data deviates from clean data.
Returns: Dict{String, Dict{Float64, Dict{Int, Int}}}
"""
function calc_deviation_indices(results_all::Dict; threshold=0.01, noise_floor=1e-15)
    dev_dict = Dict()

    for target_cutoff in keys(results_all)
        cutoff_data = results_all[target_cutoff]
        
        # clean case
        if !haskey(cutoff_data, 0.0)
            @warn "No σ=0.0 reference found for cutoff $target_cutoff. Skipping."
            continue
        end
        
        results_clean = cutoff_data[0.0]
        all_sigma_deviations = Dict{Float64, Dict{Int, Int}}()
        
        for σ in keys(cutoff_data)
            # skip clean case
            σ == 0.0 && continue
            
            results_disordered = cutoff_data[σ]
            deviation_indices = Dict{Int, Int}()
            
            available_Ns = intersect(keys(results_clean), keys(results_disordered))
            
            for N in sort(collect(available_Ns))
                v_clean = sort(results_clean[N], rev=true)
                v_disorder = sort(results_disordered[N], rev=true)
                
                max_i = min(length(v_clean), length(v_disorder))
                dev_idx = max_i 
                
                for i in 1:max_i
                    val_c = v_clean[i]^2
                    val_d = v_disorder[i]^2
                    
                    # deviation logic
                    abs_diff = abs(val_c - val_d)
                    
                    if val_c > noise_floor && val_d > noise_floor
                        rel_diff = abs_diff / max(val_c, val_d)
                        if rel_diff > threshold
                            dev_idx = i
                            break
                        end
                    elseif abs_diff > noise_floor
                        # If one is near zero and the other isn't, use absolute
                        dev_idx = i
                        break
                    end
                end
                deviation_indices[N] = dev_idx
            end
            all_sigma_deviations[σ] = deviation_indices
        end
        dev_dict[target_cutoff] = all_sigma_deviations
    end
    
    return dev_dict
end

deviation_indices = calc_deviation_indices(results, threshold=0.01, noise_floor=1e-12)

In [ ]:
"""
Splits the Schmidt spectrum into head and tail based on the deviation index.
- head_data: spectrum[1 : dev_idx-1]
- tail_data: spectrum[dev_idx : end]
"""
function split_spectrum_data(results_all::Dict, dev_indices::Dict)
    head_data = Dict()
    tail_data = Dict()

    for cutoff in keys(dev_indices)
        head_data[cutoff] = Dict()
        tail_data[cutoff] = Dict()
        
        for σ in keys(dev_indices[cutoff])
            head_data[cutoff][σ] = Dict()
            tail_data[cutoff][σ] = Dict()
            
            for N in keys(dev_indices[cutoff][σ])
                # Get the deviation index for this specific N and sigma
                idx = dev_indices[cutoff][σ][N]
                

                spectrum = sort(results_all[cutoff][σ][N], rev=true)
                
                # Split the data
                head_data[cutoff][σ][N] = spectrum[1:idx-1]
                tail_data[cutoff][σ][N] = spectrum[idx:end]
            end
        end
    end
    
    return head_data, tail_data
end

head_results, tail_results = split_spectrum_data(results, deviation_indices)


In [ ]:
# Squared dictionaries for both head and tail results
head_results_sq = Dict(
    cutoff => Dict(
        sigma => Dict(
            N => spectrum .^ 2 for (N, spectrum) in n_dict
        ) for (sigma, n_dict) in s_dict
    ) for (cutoff, s_dict) in head_results
)

tail_results_sq = Dict(
    cutoff => Dict(
        sigma => Dict(
            N => spectrum .^ 2 for (N, spectrum) in n_dict
        ) for (sigma, n_dict) in s_dict
    ) for (cutoff, s_dict) in tail_results
)

In [ ]:
"""
Plots the Schmidt spectrum for a specific parameter set, 
distinguishing the head and tail with colors and a vertical divider.
"""
function plot_head_tail_spectrum(head_data::Dict, tail_data::Dict, cutoff::String, σ::Float64, N::Int)

    head_v = head_data[cutoff][σ][N]
    tail_v = tail_data[cutoff][σ][N]

    head_indices = 1:length(head_v)
    tail_indices = (length(head_v) + 1):(length(head_v) + length(tail_v))
    
    p = plot(
        ylabel = "Schmidt Values \$(λ_i)\$",
        xlabel = "Index \$(i)\$",
        title = "Spectrum Split, N=$N, σ=$σ, Cutoff=$cutoff",
        yaxis = :log10, # Log scale
        legend = :topright,
        grid = :all,
        minorgrid = true
    )
    
    if !isempty(head_v)
        plot!(p, head_indices, head_v, 
              seriestype = :scatter, 
              color = :blue, 
              label = "Head spectrum", 
              markersize = 4)
    end
    
    if !isempty(tail_v)
        plot!(p, tail_indices, tail_v, 
              seriestype = :scatter, 
              color = :red, 
              label = "Tail spectrum", 
              markersize = 4)
    end
    

    v_line_pos = length(head_v) + 0.5
    vline!(p, [v_line_pos], 
           line = (:black, :dash, 1.5), 
           label = "Deviation Index")
    
    return p
end

p = plot_head_tail_spectrum(head_results_sq, tail_results_sq, "1.0e-10", 0.02, 10)
display(p)

In [ ]:
"""
Plots the head and tail of the Schmidt spectrum on two separate subplots.
- Subplot 1: Head (Blue)
- Subplot 2: Tail (Red)
"""
function plot_head_tail_separate(head_data::Dict, tail_data::Dict, cutoff::String, σ::Float64, N::Int)
    head_v = head_data[cutoff][σ][N]
    tail_v = tail_data[cutoff][σ][N]
    
    head_indices = 1:length(head_v)
    tail_indices = (length(head_v) + 1):(length(head_v) + length(tail_v))
    
    p1 = plot(
        head_indices, head_v,
        seriestype = :scatter,
        color = :blue,
        label = "Head Data",
        ylabel = "Schmidt Values \$(λ_i)\$",
        xlabel = "Index \$(i)\$",
        title = "Head Spectrum",
        yaxis = :log10,
        markersize = 4
    )
    
    p2 = plot(
        tail_indices, tail_v,
        seriestype = :scatter,
        color = :red,
        label = "Tail Data",
        ylabel = "", # Y-label is redundant if side-by-side
        xlabel = "Index \$(i)\$",
        title = "Tail Spectrum",
        yaxis = :log10,
        markersize = 4
    )
    

    combined_plot = plot(p1, p2, 
        layout = (1, 2), 
        size = (900, 400), 
        plot_title = "Spectrum Analysis, N=$N, σ=$σ, Cutoff=$cutoff",
        margin = 5Plots.mm
    )
    
    return combined_plot
end

# Example usage:
p_split = plot_head_tail_separate(head_results_sq, tail_results_sq, "1.0e-12", 0.002, 30)
display(p_split)

In [ ]:
"""
Plots the number of Schmidt coefficients (lengths) in the head and tail 
as a function of system size N for a specific sigma and cutoff.
"""
function plot_lengths_vs_N(head_data::Dict, tail_data::Dict, cutoff::String, σ::Float64)
    available_Ns = sort(collect(keys(head_data[cutoff][σ])))
    
    head_lengths = [length(head_data[cutoff][σ][N]) for N in available_Ns]
    tail_lengths = [length(tail_data[cutoff][σ][N]) for N in available_Ns]
    
    p1 = plot(
        available_Ns, head_lengths,
        seriestype = :scatterpath,
        color = :blue,
        marker = :circle,
        label = "Head Length",
        ylabel = "Number of Schmidt Values",
        xlabel = "System Size \$N\$",
        title = "Head Count vs N",
        grid = true
    )
    
    p2 = plot(
        available_Ns, tail_lengths,
        seriestype = :scatterpath,
        color = :red,
        marker = :square,
        label = "Tail Length",
        ylabel = "Number of Schmidt Values",
        xlabel = "System Size \$N\$",
        title = "Tail Count vs N",
        grid = true
    )
    
    combined_plot = plot(p1, p2, 
        layout = (1, 2), 
        size = (900, 400), 
        plot_title = "Spectrum Size Scaling, σ=$σ, Cutoff=$cutoff",
        margin = 5Plots.mm
    )


    p_both = plot(
        available_Ns, head_lengths,
        seriestype = :scatterpath,
        color = :blue,
        marker = :circle,
        label = "Head Length",
        ylabel = "Number of Schmidt Values",
        xlabel = "System Size \$N\$",
        title = "Spectrum Size Scaling, σ=$σ, Cutoff=$cutoff",
        grid = true,
        size = (800, 500),
        legend = :topleft,
        margin = 5Plots.mm
    )
    
    plot!(p_both,
        available_Ns, tail_lengths,
        seriestype = :scatterpath,
        color = :red,
        marker = :square,
        label = "Tail Length"
    )
    
    return combined_plot, p_both
end

p_lengths, p_both = plot_lengths_vs_N(head_results, tail_results, "1.0e-10", 0.02)
display(p_lengths)
display(p_both)

In [ ]:
"""
Plots the total weight (sum of squares) of the head and tail 
as a function of system size N for a specific sigma and cutoff.
"""
function plot_weights_vs_N(head_data::Dict, tail_data::Dict, cutoff::String, σ::Float64)

    available_Ns = sort(collect(keys(head_data[cutoff][σ])))
    
    head_weights = [sum(head_data[cutoff][σ][N].^2) for N in available_Ns]
    tail_weights = [sum(tail_data[cutoff][σ][N].^2) for N in available_Ns]
    
    p1 = plot(
        available_Ns, head_weights,
        seriestype = :scatterpath,
        color = :blue,
        marker = :circle,
        label = "Head Weight",
        ylabel = "Weight \$\\sum_i^k λ_i^2\$",
        xlabel = "System Size \$N\$",
        title = "Head Weight vs N",
        grid = true
    )
    

    p2 = plot(
        available_Ns, tail_weights,
        seriestype = :scatterpath,
        color = :red,
        marker = :square,
        label = "Tail Weight",
        ylabel = "Weight \$\\sum_{i>k}^N λ_i^2\$",
        xlabel = "System Size \$N\$",
        title = "Tail Weight vs N",
        yaxis = :log10, 
        grid = true
    )
    
    combined_plot = plot(p1, p2, 
        layout = (1, 2), 
        size = (900, 400), 
        plot_title = "Weight Scaling, σ=$σ, Cutoff=$cutoff",
        margin = 5Plots.mm,
    )
    
    return combined_plot
end

p_weights = plot_weights_vs_N(head_results, tail_results, "1.0e-10", 0.02)
display(p_weights)

In [ ]:
"""
Calculates the plateau level of the tail weight for each sigma and cutoff.
Returns: Dict{String, Dict{Float64, Float64}}
"""
function calculate_noise_plateaus(tail_data::Dict; num_last_points=3)
    plateau_dict = Dict()

    for cutoff in keys(tail_data)
        plateau_dict[cutoff] = Dict{Float64, Float64}()
        
        for σ in keys(tail_data[cutoff])
            available_Ns = sort(collect(keys(tail_data[cutoff][σ])))
            

            weights = [sum(tail_data[cutoff][σ][N].^2) for N in available_Ns]
            
            if length(weights) >= num_last_points
                plateau_level = mean(weights[end-(num_last_points-1):end])
            else
                plateau_level = weights[end]
            end
            
            plateau_dict[cutoff][σ] = plateau_level
        end
    end
    
    return plateau_dict
end

plateau_levels = calculate_noise_plateaus(tail_results)

In [ ]:
"""
Plots the Tail Weight Plateau against the Cutoff values for a specific σ.
Handles string keys for cutoffs and floating-point matching for σ.
"""
function plot_plateau_vs_cutoff(plateau_dict::Dict, target_sigma::Float64)
    valid_x = Float64[]
    valid_y = Float64[]
    
    for (c_str, sigma_map) in plateau_dict
        cutoff_val = parse(Float64, c_str)
        
        match_key = nothing
        for k in keys(sigma_map)
            if isapprox(k, target_sigma, atol=1e-12)
                match_key = k
                break
            end
        end
        
        if match_key !== nothing
            push!(valid_x, cutoff_val)
            push!(valid_y, sigma_map[match_key])
        end
    end

    if isempty(valid_x)
        error("No data found for sigma = $target_sigma.")
    end
    
    p_idx = sortperm(valid_x)
    x_sorted = valid_x[p_idx]
    y_sorted = valid_y[p_idx]
    
    p = plot(
        x_sorted, y_sorted,
        xaxis = :log10, 
        yaxis = :log10,
        seriestype = :scatterpath,
        marker = :circle,
        markersize = 5,
        color = :blue,
        linewidth = 2,
        xlabel = "Truncation value in DMRG",
        ylabel = "Tail Weight Plateau",
        title = "Tail Weight Plateau for DMRG cutoff, σ = $target_sigma",
        label = "Data (σ = $target_sigma)",
        grid = true,
        minorgrid = true,
        legend = :bottomright
    )
    
    return p
end

p_plateau = plot_plateau_vs_cutoff(plateau_levels, 0.002)
display(p_plateau)

In [ ]:
"""
Analyzes the convergence of Entanglement Entropy (EE) by comparing lower-precision 
truncation results against the highest-precision reference available.

# Arguments
- `results_dict::Dict`: The master dictionary containing spectrum data for various cutoffs and σ values.
- `target_sigma::Float64`: The specific value of σ to analyze.

# Returns
- `p`: A Plots object showing the mean relative error across system sizes N on a log-log scale.
"""
function analyze_convergence_vs_cutoff(results_dict::Dict, target_sigma::Float64)
    cutoff_keys = collect(keys(results_dict))
    numeric_vals = parse.(Float64, cutoff_keys)
    sp = sortperm(numeric_vals)
    sorted_strs = cutoff_keys[sp]
    
    # reference (the smallest cutoff value)
    ref_str = sorted_strs[1]
    
    get_s(d, s) = (for k in keys(d); if isapprox(k, s, atol=1e-10) return d[k] end; end; return nothing)
    
    ref_data = get_s(results_dict[ref_str], target_sigma)
    if ref_data === nothing
        error("Target sigma $target_sigma not found in reference cutoff $ref_str.")
    end

    # calculation for EE, Schmidt coefficients (val) are squared to get density matrix eigenvalues
    calc_ee(λs) = -sum([(val^2) * log(val^2) for val in λs if val > 1e-18])

    x_vals = Float64[]
    y_vals = Float64[]
    common_Ns = keys(ref_data)

    for s_str in sorted_strs
        # We skip the reference itself (error relative to self is 0, which breaks log scales)
        if s_str == ref_str continue end
        
        data = get_s(results_dict[s_str], target_sigma)
        if data !== nothing
            errs = [abs(calc_ee(data[N]) - calc_ee(ref_data[N])) / abs(calc_ee(ref_data[N])) 
                    for N in common_Ns if haskey(data, N)]
            
            m_err = mean(errs)
            if m_err > 0
                push!(x_vals, parse(Float64, s_str))
                push!(y_vals, m_err)
            end
        end
    end

    if isempty(y_vals)
        @warn "No valid comparison data found for sigma $target_sigma."
        return nothing
    end

    p = plot(x_vals, y_vals, 
        xaxis = (:log10, "DMRG Truncation Cutoff (ε)"), 
        yaxis = (:log10, "Mean Relative Error in EE"),
        marker = (:circle, 6, 0.8, :white, stroke(2, :blue)),
        line = (:solid, 2, :blue),
        xformatter = x -> @sprintf("%.0e", x),
        yformatter = y -> @sprintf("%.0e", y),
        title = "Convergence Analysis (σ = $target_sigma)",
        label = "Error relative to $(ref_str)",
        legend = :topleft,
        grid = :both,
        minorgrid = true,
        size = (700, 450)
    )
    
    return p
end

p_conv = analyze_convergence_vs_cutoff(results, 0.0002)
display(p_conv)

In [ ]:
"""
Analyzes physical errors in energy observables by examining:
1. Precision Error: Convergence of ground state energy as truncation ε → 0.
2. Finite-Size Error: Raw scaling of energy per site (E/N) vs 1/N.

# Arguments
- `energy_dict::Dict`: Dictionary of energy values [cutoff][sigma][N].
- `target_sigma::Float64`: The disorder strength σ to analyze.

# Returns
- `p1`: Plot object for Precision Convergence (log-log).
- `p2`: Plot object for Finite Size Data.
"""
function analyze_energy_errors(energy_dict::Dict, target_sigma::Float64)
    all_keys = collect(keys(energy_dict))
    numeric_cutoffs = [k isa String ? parse(Float64, k) : Float64(k) for k in all_keys]
    
    sp = sortperm(numeric_cutoffs)
    sorted_strs = all_keys[sp]
    sorted_vals = numeric_cutoffs[sp]
    
    # Reference is the highest precision (smallest ε, e.g., 1.0e-14)
    ref_key = sorted_strs[1]
    
    # Robust sigma lookup helper
    get_sigma_map(d, s) = begin
        for k in keys(d)
            if isapprox(k, s, atol=1e-10) return d[k] end
        end
        return nothing
    end

    ref_data = get_sigma_map(energy_dict[ref_key], target_sigma)
    if ref_data === nothing
        @error "Sigma $target_sigma not found in reference cutoff $ref_key"
        return nothing, nothing
    end

    x_conv = Float64[]
    y_err = Float64[]
    common_Ns = keys(ref_data)

    for (i, c_key) in enumerate(sorted_strs)
        if c_key == ref_key continue end # Skip comparing ref to itself
        
        data = get_sigma_map(energy_dict[c_key], target_sigma)
        if data !== nothing
            # Relative error calculation: |E - E_ref| / |E_ref|
            errs = [abs(data[N] - ref_data[N]) / abs(ref_data[N]) 
                    for N in common_Ns if haskey(data, N)]
            
            if !isempty(errs)
                push!(x_conv, sorted_vals[i])
                push!(y_err, mean(errs))
            end
        end
    end

    p1 = plot(x_conv, y_err, 
        xaxis = (:log10, "DMRG Truncation Cutoff (ε)"), 
        yaxis = (:log10, "Rel. Energy Error |ΔE/E|"),
        marker = (:circle, 6, :white, stroke(2, :red)),
        line = (:dash, 1, :red),
        xformatter = x -> @sprintf("%.0e", x),
        yformatter = y -> @sprintf("%.0e", y),
        title = "Energy Precision Convergence (σ = $target_sigma)",
        label = "Mean Error vs $ref_key",
        grid = true,
        size = (700, 450)
    )

    available_Ns = sort(collect(common_Ns))
    inv_N = 1.0 ./ available_Ns
    e_per_site = [ref_data[N]/N for N in available_Ns]

    p2 = scatter(inv_N, e_per_site, 
        markershape = :square, 
        markersize = 6, 
        markercolor = :green,
        xlabel = "Inverse System Size (1/N)", 
        ylabel = "Energy per Site (E/N)",
        title = "Finite Size Data (ε = $ref_key)",
        label = "DMRG Data (σ=$target_sigma)",
        grid = true,
        size = (700, 450)
    )

    return p1, p2
end

energy_results = Dict()
for (c, d) in data_dict
    if haskey(d, "energy_results")
        energy_results[c] = d["energy_results"]
    end
end

p_prec, p_scaling = analyze_energy_errors(energy_results, 0.0002)
display(p_prec) 
display(p_scaling)